In [4]:
# ============================================================
# CELL 0 – RUN THIS FIRST IN EVERY NOTEBOOK (do not modify)
# ============================================================
import os, sys
from google.colab import drive
import torch

# Step 1: Mount Google Drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = "/content/drive/MyDrive/HateSpeech_NLP"

# Step 2: Clone or pull the Git repository
# CHANGE THIS TO YOUR ACTUAL REPO URL
REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
else:
    os.system(f"cd {REPO_PATH} && git pull origin main")

# Step 3: Install the package in editable mode
os.system(f"pip install -q -e {REPO_PATH}")

# Step 4: Add src/ to Python path
if f"{REPO_PATH}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_PATH}/src")

# Step 5: Install remaining dependencies
os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

print("✅ Environment ready. REPO_PATH:", REPO_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready. REPO_PATH: /content/hate-speech-detection


In [5]:
# ============================================================
# CELL 1 – PREPROCESSING LOGIC
# ============================================================
import re
import os
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

def clean_text(text):
    if not isinstance(text, str): return ""
    # 1. Remove URLs and HTML entities
    text = re.sub(r'http\S+|www\S+|&[a-z]+;', '', text)
    # 2. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # 3. Normalize whitespace
    # (CRITICAL: No lowercase, No accent removal as per pipeline rules)
    text = ' '.join(text.split())
    return text

def process_data(file_name):
    # Construct paths using DRIVE_ROOT and REPO_PATH setup from Cell 0
    input_path = os.path.join(DRIVE_ROOT, f"data/raw/vihsd/{file_name}.csv")
    output_path = os.path.join(DRIVE_ROOT, f"data/processed/{file_name}.parquet")

    if not os.path.exists(input_path):
        print(f"⚠️ Warning: File not found {input_path}")
        return None

    df = pd.read_csv(input_path)
    # Use 'free_text' column from the raw CSV
    df['text'] = df['free_text'].apply(clean_text)

    # Filter very short samples (at least 2 characters)
    df = df[df['text'].str.len() >= 2]

    # Save as parquet (Enterprise requirement for efficiency)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df[['text', 'label_id']].to_parquet(output_path, index=False)
    print(f"✅ Processed and saved: {output_path}")
    return df

# Execute processing for all splits
train_clean = process_data('train')
dev_clean = process_data('dev')
test_clean = process_data('test')

# Calculate Class Weights to handle data imbalance in later phases
if train_clean is not None:
    labels = train_clean['label_id'].unique()
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.sort(labels),
        y=train_clean['label_id']
    )
    class_weights_dict = dict(zip(np.sort(labels), weights))
    print(f"\n📊 Calculated Class Weights: {class_weights_dict}")

    # Decision: Save these weights to a config or result for Phase 5
    import json
    with open(os.path.join(DRIVE_ROOT, "results/class_weights.json"), 'w') as f:
        json.dump({int(k): float(v) for k, v in class_weights_dict.items()}, f)
    print("✅ Class weights saved to results/class_weights.json")

✅ Processed and saved: /content/drive/MyDrive/HateSpeech_NLP/data/processed/train.parquet
✅ Processed and saved: /content/drive/MyDrive/HateSpeech_NLP/data/processed/dev.parquet
✅ Processed and saved: /content/drive/MyDrive/HateSpeech_NLP/data/processed/test.parquet

📊 Calculated Class Weights: {np.int64(0): np.float64(0.4033414092469211), np.int64(1): np.float64(4.978816199376947), np.int64(2): np.float64(3.1263693270735526)}
✅ Class weights saved to results/class_weights.json
